In [15]:
!pip install ultralytics gradio twilio -q

from ultralytics import YOLO
import cv2
import numpy as np
from IPython.display import display, Image, Video
import gradio as gr
from twilio.rest import Client
import smtplib
from email.mime.text import MIMEText
print("✅ Setup complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 25.4 MB/s eta 0:00:00
✅ Setup complete!


In [16]:

model = YOLO("yolo11s-pose.pt")

print("✅ Pose model loaded!")

✅ Pose model loaded!


In [17]:
def is_falling(keypoints):
    if len(keypoints) < 17:
        return False
    head_y = keypoints[0][1].item()
    ankle_y = (keypoints[15][1].item() + keypoints[16][1].item()) / 2
    shoulder_y = (keypoints[11][1].item() + keypoints[12][1].item()) / 2

    if abs(head_y - ankle_y) < 85 and abs(shoulder_y - ankle_y) < 130:
        return True
    return False

def detect_fall(image):
    results = model.predict(image, conf=0.5, verbose=False)
    annotated = results[0].plot()
    fall_detected = False

    for result in results:
        if result.keypoints is not None:
            kpts = result.keypoints.data[0]
            if is_falling(kpts):
                fall_detected = True
                cv2.putText(annotated, "🚨 FALL DETECTED!", (50, 100),
                          cv2.FONT_HERSHEY_SIMPLEX, 2.5, (0, 0, 255), 6)

    status = "🚨 Fall Detected!" if fall_detected else "✅ No Fall"
    return annotated, status

In [18]:
def video_stream_detection():
    cap = cv2.VideoCapture(0)  # Webcam
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        annotated, status = detect_fall(frame)
        cv2.putText(annotated, status, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.imshow("Fall Detection", annotated)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()

In [19]:
def full_detection(image):
    annotated, status = detect_fall(image)
    if "Fall Detected" in status:
        # send_sms_alert()
        # send_email_alert()
        pass
    return annotated, status

interface = gr.Interface(
    fn=full_detection,
    inputs=gr.Image(type="pil", label="Upload Image or Use Webcam"),
    outputs=[
        gr.Image(type="pil", label="Detection Result"),
        gr.Textbox(label="Status")
    ],
    title="🛡️ Advanced Fall Detection System",
    description="Real-time fall detection with SMS/Email alerts and video support.",
    live=True
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5f84f96add2bc7a967.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [20]:
# Export to TFLite for mobile deployment
model.export(format="tflite")
print("✅ Model exported to TFLite for Android/iOS deployment!")

WARNING ⚠️ format='tflite' is deprecated as of 8.4.83 and has been replaced by the unified Google LiteRT format. Exporting format='litert' instead. See https://docs.ultralytics.com/integrations/litert/
Ultralytics 8.4.83 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11s-pose summary (fused): 109 layers, 9,902,940 parameters, 0 gradients, 23.1 GFLOPs

PyTorch: starting from 'yolo11s-pose.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 56, 8400) (19.4 MB)
requirements: Ultralytics requirements ['litert-torch>=0.9.0', 'ai-edge-litert>=2.1.4'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 66 packages in 1.05s
Prepared 15 packages in 7.79s
Uninstalled 3 packages in 60ms
Installed 15 packages in 94ms
 + ai-edge-litert==2.1.5
 + ai-edge-quantizer==0.7.0
 + backports-stren

/usr/local/lib/python3.12/dist-packages/torchao/quantization/quant_api.py:1745: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.



LiteRT: starting export with litert_torch 0.9.1...


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

...

(00:30) [ DONE] LiteRT-Torch Convert > Run LiteRT Converter Passes (+00:00)

(00:30) [ DONE] LiteRT-Torch Convert (+00:30)

(00:00) [START] Write Model to yolo11s-pose.tflite

(00:00) [ DONE] Write Model to yolo11s-pose.tflite (+00:00)

LiteRT: export success ✅ 54.4s, saved as 'yolo11s-pose.tflite' (38.2 MB)

Export complete (56.3s)
Results saved to /content/yolo11s-pose.tflite
Predict:         yolo predict task=pose model=yolo11s-pose.tflite imgsz=640 
Validate:        yolo val task=pose model=yolo11s-pose.tflite imgsz=640 data=/ultralytics/ultralytics/cfg/datasets/coco-pose.yaml  
Visualize:       https://netron.app
✅ Model exported to TFLite for Android/iOS deployment!
